In [ ]:
from utils import btree
from utils import pixcavation_api

In [ ]:
def identify_char_at(char_index, line_index, char_width, char_height, ocr_btree):
    x_offset = char_index * char_width
    y_offset = line_index * char_height

    curr_node = ocr_btree

    while not curr_node.is_leaf():
        x, y = curr_node.value
        val = api.reveal_one_pixel(x + x_offset, y + y_offset)["revealed"][0]["pixelVal"]
        if val == 1:
            # go left
            curr_node = curr_node.left
        else:
            # got right
            curr_node = curr_node.right

    return curr_node.value

In [ ]:
def read_scripture(txt_line_len, txt_line_count, char_width, char_height, ocr_btree):
    txt = ""
    for i in range(txt_line_count):
        for j in range(txt_line_len):
            char = identify_char_at(
                char_index=j,
                line_index=i,
                char_width=char_width,
                char_height=char_height,
                ocr_btree=ocr_btree
            )
            txt += char
        txt += "\n"
    return txt


In [ ]:
# Loading ocr btree
pixel_perfect_ocr_btree = btree.load_btree_json("../out/ocr_btree.json")
# btree.print_btree(ocr_btree)

# grabbing a client game api instance
api = pixcavation_api.PixcavationAPI(base_url="http://localhost:2026")
api.exists()

# creating a new game (asking for a fresh scripture)
resp = api.new_game()
# print(resp)

# grabbing metadata
# (dynamic here cause I wanna be able to re-configure the game without revisiting the solution)
state = resp["state"]
text_line_length = state["textLineLength"]
text_line_count = state["textLineCount"]
character_width = state["width"] / text_line_length
character_height = state["height"] / text_line_count

# reading the scripture :))
scripture = read_scripture(
    txt_line_len=text_line_length,
    txt_line_count=text_line_count,
    char_width=character_width,
    char_height=character_height,
    ocr_btree=pixel_perfect_ocr_btree
)

# printing scripture
print(f"Scripture: {scripture}")

# submitting it so that I don't have to type it
flag = api.submit_guess("".join(scripture.split("\n")))["state"]["flag"]

# voila!
print(f"Flag: {flag}")